# Prophet R04: resumable pilot
Pinned training source, GPU gate, audited corpus and vocabulary, whole-document validation.
The two arms and three seeds remain experiments. Model quality is measured separately.
Train and monitor on local Colab storage. Copy a verified snapshot to Drive after each
session; keep the VM until remote flushing succeeds. Review the resume source below.


In [ ]:
# Mount access explicitly approved by the user on 2026-09-19.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
persistent = Path('/content/drive/MyDrive/Prophet_AGI/R04')
persistent.mkdir(parents=True, exist_ok=True)


In [ ]:
assert 'r04_process' not in globals() or r04_process.poll() is not None
assert 'snapshot_process' not in globals() or snapshot_process.poll() is not None
import os, subprocess, sys, shutil
from pathlib import Path
repo = Path('/content/Prophet_AGI')
revision = 'e5720d0b774977455b1920a6f67b5df078377f11'
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/speed25200-cyber/Prophet_AGI.git', str(repo)], check=True)
subprocess.run(['git', 'fetch', 'origin', 'claude/prophet-v03-memory-context'], cwd=repo, check=True)
subprocess.run(['git', 'checkout', revision], cwd=repo, check=True)
os.environ['TRITON_F32_DEFAULT'] = 'tf32x3'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)+'[dev,gpu]', 'datasets==5.0.1', 'huggingface-hub==1.32.0'], check=True)
gate_path = Path('/content/gpu-gate.log')
with gate_path.open('w') as gate_log:
    gate = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_gpu.py', '-v', '--tb=short'], cwd=repo, stdout=gate_log, stderr=subprocess.STDOUT, timeout=900)
print(gate_path.read_text())
gate.check_returncode()
corpus = repo / 'data/fineweb-pilot-v1'
cache = persistent / 'corpus-v1'
if not corpus.exists():
    if cache.exists():
        shutil.copytree(cache, corpus)
    else:
        subprocess.run([sys.executable, 'scripts/prepare_pilot.py', '--out', str(corpus)], cwd=repo, check=True)
        subprocess.run([sys.executable, 'scripts/train_tokenizer.py', '--data-root', str(corpus/'train'), '--out', str(corpus/'tokenizer.json'), '--vocab-size', '32768', '--max-docs', '10000'], cwd=repo, check=True)
        subprocess.run([sys.executable, 'scripts/run_r04_pilot.py', '--corpus', str(corpus), '--variant', 'loop', '--seed', '0', '--out', str(persistent/'loop-seed0'), '--dry-run'], cwd=repo, check=True)
        shutil.copytree(corpus, persistent/'corpus-v1.building')
        (persistent/'corpus-v1.building').rename(cache)
print('R04_SETUP_VERIFIED', revision, flush=True)


In [ ]:
# Analysis helpers do not change the frozen training checkout.
helper_revision = 'a08a15fb493beabb9969c405f01380b28ca185ec'
subprocess.run(['git', 'fetch', 'origin', helper_revision], cwd=repo, check=True)
helper_root = Path('/content/r04-tools')
helper_root.mkdir(exist_ok=True)
for name in ('audit_r04_checkpoint.py', 'snapshot_r04.py'):
    (helper_root/name).write_bytes(subprocess.check_output(['git','show',helper_revision+':scripts/'+name], cwd=repo))
sys.path.insert(0, str(helper_root))
from audit_r04_checkpoint import audit_checkpoint
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip() == revision


In [ ]:
# Choose the arm/seed and its verified source explicitly. None means a new run.
variant, seed, session_steps = 'loop', 0, 128
resume_from = persistent / 'snapshots/step-001024-seed0/loop-seed0'
assert variant in ('loop', 'plain') and seed in (0, 1, 2)
assert 'r04_process' not in globals() or r04_process.poll() is not None
assert 'snapshot_process' not in globals() or snapshot_process.poll() is not None
import json, time
working_run = Path('/content/r04-working') / f'{variant}-seed{seed}'
if not working_run.exists() and resume_from is not None:
    reports = sorted(resume_from.glob('evaluation-step-*.json'))
    assert reports, 'No completed evaluation in the chosen resume source'
    restored_step = json.loads(reports[-1].read_text())['step']
    before = audit_checkpoint(resume_from, restored_step)
    assert before['variant'] == variant and before['seed'] == seed
    assert before['all_model_optimizer_tensors_finite'] and before['skipped_nonfinite'] == 0
    staging = working_run.with_name(working_run.name+'.restoring')
    assert not staging.exists(), 'Inspect the earlier partial restore before retrying'
    shutil.copytree(resume_from, staging, ignore=shutil.ignore_patterns('SNAPSHOT_COMPLETE.json'))
    assert audit_checkpoint(staging, restored_step) == before
    staging.rename(working_run)
if working_run.exists():
    protocol = json.loads((working_run/'protocol.json').read_text())
    assert protocol['variant'] == variant and protocol['seed'] == seed
print('LOCAL_WORKING_RUN', working_run, flush=True)


In [ ]:
# A bounded part of the fixed 4096-step experiment. Repeat to resume locally.
assert 'r04_process' not in globals() or r04_process.poll() is not None
assert 'snapshot_process' not in globals() or snapshot_process.poll() is not None
log_path = Path('/content') / f'{variant}-seed{seed}-session-{time.time_ns()}.log'
remote_flushed = False
with log_path.open('w', buffering=1) as log_stream:
    r04_process = subprocess.Popen([sys.executable, '-u', 'scripts/run_r04_pilot.py', '--corpus', str(corpus), '--variant', variant, '--seed', str(seed), '--out', str(working_run), '--max-session-steps', str(session_steps), '--session-minutes', '45'], cwd=repo, stdout=log_stream, stderr=subprocess.STDOUT, start_new_session=True)
print('R04_PROCESS', r04_process.pid, 'LOG', str(log_path), flush=True)
# To stop early, run r04_process.terminate(), then wait for EXIT 0.
# The signal requests a completed-step checkpoint; notebook interrupts are isolated.


In [ ]:
print('EXIT', r04_process.poll())
print(log_path.read_text()[-16000:])


In [ ]:
# Run only after full evaluation has completed. Preserve each earlier snapshot.
assert r04_process.poll() == 0
assert 'snapshot_process' not in globals() or snapshot_process.poll() is not None
reports = sorted(working_run.glob('evaluation-step-*.json'))
assert reports
completed_step = json.loads(reports[-1].read_text())['step']
shutil.copyfile(log_path, working_run/log_path.name)
snapshot_destination = persistent / 'snapshots' / f'step-{completed_step:06d}-seed{seed}-{time.time_ns()}' / f'{variant}-seed{seed}'
snapshot_log = Path('/content') / f'snapshot-{time.time_ns()}.log'
with snapshot_log.open('w') as stream:
    snapshot_process = subprocess.Popen([sys.executable, '-u', str(helper_root/'snapshot_r04.py'), '--run', str(working_run), '--step', str(completed_step), '--destination', str(snapshot_destination)], stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
print('SNAPSHOT_STARTED', snapshot_process.pid, snapshot_destination, flush=True)


In [ ]:
print('SNAPSHOT_EXIT', snapshot_process.poll())
print(snapshot_log.read_text()[-8000:])


In [ ]:
# Retain experiment reports before releasing the VM. A filesystem marker alone
# does not prove remote persistence. A failed flush must leave the VM available.
assert r04_process.poll() == 0 and snapshot_process.poll() == 0
marker = json.loads((snapshot_destination/'SNAPSHOT_COMPLETE.json').read_text())
assert marker['complete'] and marker['checkpoint']['step'] == completed_step
assert os.path.ismount('/content/drive'), 'Drive is unmounted; a flush would be a no-op'
from google.colab import drive
remote_flushed = False
drive.flush_and_unmount(timeout_ms=300000)
remote_flushed = True
print('REMOTE_FLUSH_COMPLETED', snapshot_destination, flush=True)


In [ ]:
# Separate final step, after recording the completed snapshot and results.
assert remote_flushed and r04_process.poll() == 0 and snapshot_process.poll() == 0
from google.colab import runtime
runtime.unassign()
